# M6-B1 — Analyse de dérive (à compléter)

In [12]:
import os

print(os.getcwd())

C:\Users\seugn\PycharmProjects\M6-B1-alex-frank-etienne\notebooks


## 1. Exploration — distributions référence vs prod

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, chi2_contingency

reference = pd.read_csv("../data/reference_set.csv")
prod = pd.read_csv("../data/prod_3months.csv")


def psi(ref, cur, n_bins=10, eps=1e-6):
    ref = ref.dropna()
    cur = cur.dropna()

    edges = np.unique(np.quantile(ref, np.linspace(0, 1, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf

    p_ref = np.histogram(ref, edges)[0] / len(ref)
    p_cur = np.histogram(cur, edges)[0] / len(cur)

    p_ref, p_cur = p_ref + eps, p_cur + eps
    p_ref, p_cur = p_ref / p_ref.sum(), p_cur / p_cur.sum()

    return float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))


for column in ["int_rate", "loan_amnt"]:
    ref = reference[column]
    cur = prod[column]

    print(column)
    print("PSI :", round(psi(ref, cur), 3))
    print("KS p-value :", ks_2samp(ref.dropna(), cur.dropna()).pvalue)
    print()

grades = sorted(
    set(reference["grade"].dropna().unique())
    | set(prod["grade"].dropna().unique())
)

ref_counts = reference["grade"].value_counts().reindex(grades, fill_value=0)
prod_counts = prod["grade"].value_counts().reindex(grades, fill_value=0)

table = pd.DataFrame({
    "reference": ref_counts,
    "prod": prod_counts
})

print(table)

chi2, p_value, dof, expected = chi2_contingency(table.T)

print("Chi² :", chi2)
print("p-value :", p_value)

print("Grades normalized %")
grade_compare = pd.DataFrame({
    "reference": reference["grade"].value_counts(normalize=True),
    "prod": prod["grade"].value_counts(normalize=True)
}) * 100

grade_compare = grade_compare.round(2)

print(grade_compare)


int_rate
PSI : 0.444
KS p-value : 4.627646432732096e-65

loan_amnt
PSI : 0.003
KS p-value : 0.9890912120116152

       reference  prod
grade                 
A            275   451
B            455   767
C            411   819
D            208   537
E            109   282
F             24    99
G             18    45
Chi² : 41.40073384679395
p-value : 2.414183658745721e-07


## 2. Détection statistique — PSI · KS · Chi²

In [ ]:
# TODO


## 3. Calibration en exploitation

In [ ]:
# TODO


## 4. Performance dans le temps

In [ ]:
# TODO


## 5. Diagnostic & recommandation

In [ ]:
# TODO
